# Single Chunk Debug Notebook

Development sandbox for stepping through a **single chunk** of a textbook chapter.
Uses the `%load` bridge pattern — implementation lives in `src/`, this notebook is for debugging.

**Pipeline:** `check_database` → `draft` → `judge` → `revise` (if needed) → `save_to_db`

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import hashlib
from pathlib import Path
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
sys.path.insert(0, str(Path().resolve().parent / 'src'))

from note_taker.database import DatabaseManager
from note_taker.processing import parse_markdown_chunks
from note_taker.pipeline.nodes import check_database_node, draft_node, judge_node, revise_node, save_to_db_node
from note_taker.pipeline.state import GraphState
from note_taker.models import FinalArtifactV1

DatabaseManager._instance = None
db = DatabaseManager('dev_notes.db')
db.ensure_database()
rprint('[green]Setup complete[/green]')

## Load Chapter File

In [ ]:
DATA_DIR = Path().resolve().parent / 'data' / 'raw' / 'building-applications-with'
target_file = DATA_DIR / 'chapter_005.md'

rprint(f"Target file exists: [bold]{'✓' if target_file.exists() else '✗'}[/bold]")
rprint(f"File: [cyan]{target_file.name}[/cyan]")

## Chunk the Chapter

In [ ]:
chunks = parse_markdown_chunks(str(target_file))
rprint(f"Total chunks found: [bold green]{len(chunks)}[/bold green]")

for i, c in enumerate(chunks):
    rprint(f"  [{i:03d}] {c['title']}")

## Pick a Single Chunk

In [ ]:
chunk_index = 1  # Change this to select a different chunk
selected_chunk = chunks[chunk_index]

rprint(f"Selected: [bold]{selected_chunk['title']}[/bold]")
rprint(f"Content length: {len(selected_chunk['content'])} chars")
rprint('---')
rprint(selected_chunk['content'][:300] + '...')

## Node: check_database
Check if this chunk has already been processed. Sets `skip_processing` in `GraphState`.

In [ ]:
source_hash = hashlib.sha256(selected_chunk['content'].encode('utf-8')).hexdigest()
state: GraphState = {
    "chunk_id": f"BuildingAIAgents:Chapter5:{chunk_index:03d}",
    "source_content": selected_chunk['content'],
    "source_hash": source_hash,
    "force_refresh": True,  # Set False to test skip logic
    "revision_count": 0,
    "artifact": None,
    "skip_processing": False
}

new_state = check_database_node(state, db_manager=db)
rprint(f"Chunk ID   : {state['chunk_id']}")
rprint(f"Source hash: {source_hash[:16]}...")
rprint(f"Skip processing? [bold]{'Yes ⏭' if new_state['skip_processing'] else 'No 🔄'}[/bold]")

## Node: draft
Generate the initial outline and Q&A pairs. Calls the Groq API.

In [ ]:
rprint('[yellow]▶ Running draft_node...[/yellow]')
draft_result = draft_node(new_state)
artifact = draft_result['artifact']
new_state['artifact'] = artifact

rprint(f'[green]✓ Draft complete[/green]')
rprint(f'  Outline items: {len(artifact.outline)}')
rprint(f'  Q&A pairs: {len(artifact.qa_pairs)}')
rprint()
for i, qa in enumerate(artifact.qa_pairs):
    rprint(f'  [{i}] Q: {qa.question}')
    rprint(f'      A: {qa.answer[:120]}...')
    rprint()

## Node: judge
Score each Q&A pair on accuracy, clarity, and recall-worthiness (0.0–1.0).

In [ ]:
rprint('[yellow]▶ Running judge_node...[/yellow]')
judge_result = judge_node(new_state)
artifact = judge_result['artifact']
new_state['artifact'] = artifact

failing = [qa for qa in artifact.qa_pairs if qa.judge_score is not None and qa.judge_score < 0.7]
rprint(f'[green]✓ Judge complete[/green]')
rprint(f'  Passing: {len(artifact.qa_pairs) - len(failing)} | Failing: {len(failing)}')
rprint()
for i, qa in enumerate(artifact.qa_pairs):
    icon = '✓' if qa.judge_score and qa.judge_score >= 0.7 else '✗'
    rprint(f'  [{i}] {icon} score={qa.judge_score}  feedback={qa.judge_feedback}')

## Node: revise (if needed)
Rewrite Q&A pairs that scored < 0.7 based on judge feedback.

In [ ]:
if failing:
    rprint('[yellow]▶ Running revise_node...[/yellow]')
    revise_result = revise_node(new_state)
    artifact = revise_result['artifact']
    new_state['artifact'] = artifact
    rprint(f'[green]✓ Revision complete (count: {revise_result["revision_count"]})[/green]')
    for i, qa in enumerate(artifact.qa_pairs):
        rprint(f'  [{i}] Q: {qa.question}')
else:
    rprint('[green]All pairs passed — no revision needed.[/green]')

## Save to Dev DB
Persist the final `FinalArtifactV1` to the development SQLite database.

In [ ]:
save_to_db_node(new_state, db_manager=db)
rprint(f'[green]✓ Artifact saved as {new_state["chunk_id"]}[/green]')

# Verify round-trip
retrieved = db.get_artifact(new_state['chunk_id'])
rprint(f'[green]✓ Retrieved from DB: {len(retrieved.qa_pairs)} Q&A pairs, {len(retrieved.outline)} outline items[/green]')